# Remaining PJM regions: attention-head fault tolerance

Replication of `transformers_EXP_B (1)` on every PJM hourly-load region **other than AEP**, which is the region used in the source paper.

## Architecture: Tiny GPT-2, decoder-only

This notebook is a derived copy of the matching notebook in
`notebooks/base_transformer/`. Every dataset, fault-injection and analysis
cell is identical; the only change is the architecture, taken from
`C.TINY_GPT2_PARAM` - the Tiny GPT-2 of *Dropout and the Outliers*
(Hezbri et al., Table 4): **6 blocks x 6 heads = 36 attention heads**,
`d_model` 384, `d_ff` 1536, pre-normalisation, learnable absolute positional
embeddings, **no biases anywhere**, **causal (left-to-right) attention**, GELU.

| | Base family | Tiny ViT family | This family |
|---|---|---|---|
| Layers x heads | 4 x 8 = 32 | 6 x 6 = 36 | **6 x 6 = 36** |
| Width / FFN | 256 / 1024 | 384 / 1536 | **384 / 1536** |
| Normalisation | post-norm | pre-norm | **pre-norm** |
| Positional encoding | sinusoidal | learnable | **learnable** |
| q/k/v biases | present | absent | **absent** |
| Other biases | present | present | **absent** |
| Attention | bidirectional | bidirectional | **causal** |
| Read-out token | CLS at position 0 | CLS at position 0 | **CLS at last position** |
| Stack parameters | 3,159,040 | 10,639,872 | **10,621,440** |

**Against the Tiny ViT family this is very nearly a single-factor contrast.**
The paper gives its Tiny GPT-2 and Tiny ViT the same six-block, six-head,
width-384 shape, so only two things differ here, and the first is nearly free:
dropping the remaining biases removes 18,432 of 10,639,872 stack parameters
(0.17%). The substantive variable is **causal versus bidirectional attention**.
That makes the Tiny ViT -> Tiny GPT-2 comparison a much cleaner attribution
than the base -> Tiny ViT one, where seven choices move at once.

Two consequences of causal masking to keep in mind when reading the results:

1. **The read-out token moved.** Under a causal mask a CLS token at position 0
   could attend only to itself, so it is appended at the end instead and
   `encode()` reads the last position. Everything else in the pipeline is
   position-agnostic.
2. **`max_cumulative_attention` is not comparable across families.**
   `attention_stats.csv` sums each key's attention over all queries; under a
   causal mask key `j` is visible only to queries `j` and later, so its ceiling
   falls from `L` to `L - j` and the maximum over keys is structurally biased
   toward early positions. Compare that column within this family only.
   (`mean_cumulative_attention` is exactly 1.0 in every family by construction
   - row sums of 1 over `L` queries and `L` keys - so it carries no
   information either way.) The criticality, degradation and safe-region
   results are unaffected: they are defined on task metrics, not attention mass.

The training recipe, the fault protocol, the criticality definitions and the
evaluation subsets are deliberately unchanged, so the degradation numbers are
comparable with the other two families run for the same dataset.

The preprocessing, architecture, training configuration, fault protocol and safe-region analysis are all as in EXP_B; only the series changes. Eleven regions with one fixed architecture answer a question the source study cannot: is the critical head's layer a property of the *architecture* or of the *series*?

**Model**: fixed *Base* configuration - 4 encoder layers x 8 heads = **32 attention heads**, `d_model` 256, `d_ff` 1024, SiLU, scalar input projection, CLS-token regression head, one-step-ahead forecast from a 20-hour window.

**Research questions** (as in the source study)

- **RQ1** Do Transformer models contain critical and redundant attention heads?
- **RQ2** Is degradation explained by the number of failed heads alone, or also by which heads fail?
- **RQ3** How do the fault-tolerance results compare across datasets and modalities, given one fixed architecture?

**Hypotheses**

- **H1** Attention-head criticality is non-uniform across the architecture.
- **H2** Layer-wise fragility is non-monotonic and task-dependent.
- **H3** Degradation increases with the number of failed attention heads.
- **H4** Criticality-aware fault severity adds information beyond the raw fault count.
- **H5** Configurations containing more critical heads are likelier to exceed an acceptable degradation threshold.

## Imports

In [1]:
import json, pathlib, sys, time

import numpy as np
import pandas as pd
import torch

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from faulttol import config as C
from faulttol import multihead, plots, saferegion, single_head
from faulttol.injection import collect_attention_stats_per_head, get_all_heads
from faulttol.runtime import configure_threads, device_report, select_device
from faulttol.training import make_loader

print("torch", torch.__version__, "| threads", configure_threads(),
      "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
print("root:", ROOT)

torch 2.14.0+cu126 | threads 8 | cuda build 12.6 | cuda available True
root: D:\Laboratory\Transformers fault tollerance


## Global variables

In [2]:
RESULTS_DIR = ROOT / "results" / "tiny_gpt2" / "pjm_regions"
DATA_DIR = ROOT / "data" / "pjm"
# NVIDIA GPU when one is visible; FT_DEVICE=cpu reproduces the CPU-only run
DEVICE = select_device()

# Every PJM region except AEP, which is the region used in the source paper.
# Set FT_REGIONS to a comma-separated list to override, e.g. "PJME,COMED".
ALL_REGIONS = ["PJME", "COMED", "DAYTON", "DOM", "DUQ", "PJMW",
               "FE", "NI", "DEOK", "EKPC", "PJM_Load"]
import os
REGIONS = [r for r in os.environ.get("FT_REGIONS", ",".join(ALL_REGIONS)).split(",") if r]

WINDOW_SIZE = 20
N_SAMPLED_WINDOWS = C._env_int("FT_WINDOWS", 50_000)
N_VAL, N_TEST = 4_000, 10_000

MODEL_CONFIG = dict(C.TINY_GPT2_PARAM, activation="gelu")

TRAIN_CONFIG = {
    "loss": "MSELoss",
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "batch_size": 128,
    "epochs": C._env_int("FT_EPOCHS", 50),
    "patience": C._env_int("FT_PATIENCE", 8),
    "grad_clip": 1.0,
    "selection_metric": "val_mae",
}

# degradation variables (as in EXP_B)
CRIT_COL = "mean_harmful_error_change"
DEG_COL = "delta_mae"
SAFE_COL = "delta_mape"
DEG_LABEL = "delta MAE"
SAFE_LABEL = "delta MAPE (pp)"

print("device:", device_report(DEVICE))
print("regions:", REGIONS)
print("windows %d | epochs %d" % (N_SAMPLED_WINDOWS, TRAIN_CONFIG["epochs"]))

device: cuda:0 NVIDIA GeForce RTX 4070 Laptop GPU | 8.0 GB | sm_89 | torch cuda 12.6 | tf32 off
regions: ['PJME', 'COMED', 'DAYTON', 'DOM', 'DUQ', 'PJMW', 'FE', 'NI', 'DEOK', 'EKPC', 'PJM_Load']
windows 50000 | epochs 50


## STEP 1: hourly load datasets

In [3]:
def load_region(region):
    df = pd.read_csv(DATA_DIR / ("%s_hourly.csv" % region))
    value_col = [c for c in df.columns if c != "Datetime"][0]
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = (df.drop_duplicates(subset="Datetime", keep="first")
            .sort_values("Datetime").reset_index(drop=True))
    values = df[value_col].astype(float).to_numpy()
    values = values[np.isfinite(values)]
    vmin, vmax = float(values.min()), float(values.max())
    return (values - vmin) / (vmax - vmin), {
        "value_col": value_col, "n_obs": int(len(values)),
        "min_mw": vmin, "max_mw": vmax,
        "start": str(df["Datetime"].iloc[0]), "end": str(df["Datetime"].iloc[-1])}

def build_windows(series, window, n_sampled, seed):
    n_windows = len(series) - window
    rng = np.random.default_rng(seed)
    chosen = rng.choice(np.arange(n_windows), size=min(n_sampled, n_windows), replace=False)
    X = np.stack([series[i:i + window] for i in chosen]).astype(np.float32)
    return X, series[chosen + window].astype(np.float32)

overview = []
for r in REGIONS:
    _, meta = load_region(r)
    overview.append({"region": r, "column": meta["value_col"], "n_obs": meta["n_obs"],
                     "start": meta["start"][:10], "end": meta["end"][:10],
                     "min_MW": meta["min_mw"], "max_MW": meta["max_mw"]})
pd.DataFrame(overview)

,region,column,n_obs,start,end,min_MW,max_MW
0,PJME,PJME_MW,145362,2002-01-01,2018-08-03,14544.0,62009.0
1,COMED,COMED_MW,66493,2011-01-01,2018-08-03,7237.0,23753.0
2,DAYTON,DAYTON_MW,121271,2004-10-01,2018-08-03,982.0,3746.0
3,DOM,DOM_MW,116185,2005-05-01,2018-08-03,1253.0,21651.0
4,DUQ,DUQ_MW,119064,2005-01-01,2018-08-03,1014.0,3054.0
5,PJMW,PJMW_MW,143202,2002-04-01,2018-08-03,487.0,9594.0
6,FE,FE_MW,62870,2011-06-01,2018-08-03,0.0,14032.0
7,NI,NI_MW,58450,2004-05-01,2011-01-01,7003.0,23631.0
8,DEOK,DEOK_MW,57735,2012-01-01,2018-08-03,907.0,5445.0
9,EKPC,EKPC_MW,45330,2013-06-01,2018-08-03,514.0,3490.0


### Min-max normalisation, sliding windows, splits

In [4]:
from faulttol.metrics import (regression_clean_metrics, regression_damage,
                              regression_outputs)
from faulttol.model import SeriesRegressor, count_params
from faulttol.training import train_regressor

def prepare_region(region):
    """Min-max normalise, window, split, and pick the fixed fault-eval subset."""
    series, meta = load_region(region)
    X, y = build_windows(series, WINDOW_SIZE, N_SAMPLED_WINDOWS, C.DATA_SEED)

    n = len(X)
    n_test = min(N_TEST, n // 5)
    n_dev = n - n_test
    n_val = min(N_VAL, max(1, n_dev // 10))
    n_train = n_dev - n_val

    Xt = torch.from_numpy(X)
    parts = {
        "X_train": Xt[:n_train], "y_train": y[:n_train],
        "X_val": Xt[n_train:n_dev], "y_val": y[n_train:n_dev],
        "X_test": Xt[n_dev:], "y_test": y[n_dev:],
    }
    rng = np.random.default_rng(C.MULTIHEAD_FAULT_SEED)
    sub = rng.choice(len(parts["X_test"]), size=min(C.N_FAULT, len(parts["X_test"])),
                     replace=False)
    parts["X_eval"] = parts["X_test"][sub].contiguous()
    parts["y_eval"] = parts["y_test"][sub]
    parts["denorm"] = lambda a: a * (meta["max_mw"] - meta["min_mw"]) + meta["min_mw"]
    parts["meta"] = meta
    return parts

print("prepared helper for", len(REGIONS), "regions")

prepared helper for 11 regions


## Fault model

**Definition 1 (attention-head crash).** A head is crashed when its output tensor is
replaced with zeros *after* the attention weights multiply V and *before* head
concatenation and the output projection `W_o`.

Faults are injected **at inference only** - the model is trained clean, then frozen.
After every faulted evaluation all masks are restored to 1.

## STEP 2: encoder-only Transformer regressor per region

In [5]:
def run_region(region):
    """Full EXP_B pipeline for one region: train, then steps 3-7."""
    global model, sh_df, mh_df, clean_eval, damage_fn, evaluate
    global weight_lookup, critical_heads, n_heads, model_summary
    global corr_rows, safe_summary, EXTRA_META, OUT, TABLES, FIGURES, MODELS
    global CRIT_COL, DEG_COL, SAFE_COL, TAUS, TAU_LABELS, MODEL_NAME

    print("\n" + "=" * 78)
    print("PJM region", region)
    print("=" * 78, flush=True)

    MODEL_NAME = "tiny_gpt2_regressor_%s" % region
    OUT = RESULTS_DIR / region
    TABLES, FIGURES, MODELS = OUT / "tables", OUT / "figures", OUT / "models"
    for d in (TABLES, FIGURES, MODELS):
        d.mkdir(parents=True, exist_ok=True)

    p = prepare_region(region)
    print("  obs=%d (%s..%s)  train=%d val=%d test=%d"
          % (p["meta"]["n_obs"], p["meta"]["start"][:10], p["meta"]["end"][:10],
             len(p["X_train"]), len(p["X_val"]), len(p["X_test"])), flush=True)

    torch.manual_seed(C.TRAIN_SEED)
    np.random.seed(C.TRAIN_SEED)
    model = SeriesRegressor(MODEL_CONFIG, max_len=WINDOW_SIZE, activation="gelu")

    hist = train_regressor(
        model, p["X_train"], p["y_train"], p["X_val"], p["y_val"], DEVICE,
        lr=TRAIN_CONFIG["learning_rate"], weight_decay=TRAIN_CONFIG["weight_decay"],
        epochs=TRAIN_CONFIG["epochs"], batch_size=TRAIN_CONFIG["batch_size"],
        patience=TRAIN_CONFIG["patience"], grad_clip=TRAIN_CONFIG["grad_clip"],
        ckpt_path=MODELS / "best.pt",
        config={"model": MODEL_CONFIG, "train": TRAIN_CONFIG, "region": region})
    pd.DataFrame(hist["history"]).to_csv(TABLES / "training_history.csv", index=False)

    denorm = p["denorm"]
    test_metrics = regression_clean_metrics(
        regression_outputs(model, p["X_test"], p["y_test"], DEVICE), denorm=denorm)

    def _evaluate():
        return regression_outputs(model, p["X_eval"], p["y_eval"], DEVICE, chunk=1024)

    def _damage(clean, fault):
        return regression_damage(clean, fault, denorm=denorm)

    evaluate, damage_fn = _evaluate, _damage
    clean_eval = regression_clean_metrics(_evaluate(), denorm=denorm)
    print("  held-out test:", test_metrics)
    print("  fixed eval subset (n=%d):" % len(p["X_eval"]), clean_eval, flush=True)

    attn = collect_attention_stats_per_head(
        model, make_loader(p["X_eval"], torch.as_tensor(p["y_eval"]), 256, shuffle=False),
        DEVICE)
    pd.DataFrame(attn).to_csv(TABLES / "attention_stats.csv", index=False)

    TAUS = [a * clean_eval["mape"] for a in C.REG_ALPHAS]
    TAU_LABELS = ["%d%%_of_clean_mape" % int(100 * a) for a in C.REG_ALPHAS]

    EXTRA_META = {
        "region": region, "series_meta": p["meta"], "window_size": WINDOW_SIZE,
        "train_config": TRAIN_CONFIG, "model_config": MODEL_CONFIG,
        "held_out_test_metrics": test_metrics, "n_params": count_params(model),
        "best_val_mae": hist["best_score"],
        "clean_mape_thresholds": dict(zip(TAU_LABELS, TAUS)),
    }
    return p

print("run_region defined")

run_region defined


## STEPS 3-6: fault injection, severity correlations, safe regions

The same shared stage functions used by the AG News and CIFAR-10 notebooks.

In [6]:
# STEPS 3-6 for one region, using the same shared stage functions as the
# AG News and CIFAR-10 notebooks.
def analyse_region(region):
    global sh_df, mh_df, weight_lookup, critical_heads, n_heads, model_summary
    global corr_rows, safe_summary

    def _save(df, name):
        df.to_csv(TABLES / name, index=False)
        return df

    # STEP 3 - single-head crash
    sh_df = single_head.run_single_head_analysis(
        model=model, evaluate=evaluate, damage_fn=damage_fn,
        criticality_col=CRIT_COL, model_name=MODEL_NAME, verbose=False)
    _save(sh_df, "single_head_damage.csv")

    layer_summary = single_head.build_layer_summary(sh_df)
    model_summary = single_head.build_model_summary(sh_df, layer_summary, MODEL_NAME, clean_eval)
    _save(layer_summary, "layer_summary.csv")
    plots.vulnerability_map(sh_df, "%s - single-head vulnerability" % MODEL_NAME,
                            FIGURES / "vulnerability_map.png")
    plots.criticality_profile(sh_df, "%s - criticality concentration" % MODEL_NAME,
                              FIGURES / "criticality_profile.png", ylabel=CRIT_COL)

    # STEP 4 - criticality, redundancy, severity weights
    weight_lookup = single_head.build_head_weight_lookup(sh_df)
    critical_heads = single_head.critical_head_set(sh_df, C.CRITICAL_TOP_FRACTION)
    n_heads = len(get_all_heads(model))
    print("  most critical head: %s (%.6f) | critical set: %s"
          % (model_summary["most_critical_head"], model_summary["max_criticality"],
             ["L%d_H%d" % h for h in critical_heads]), flush=True)

    # STEP 5 - random multi-head injection
    mh_df = multihead.run_random_multihead(
        model=model, evaluate=evaluate, damage_fn=damage_fn,
        weight_lookup=weight_lookup, critical_heads=critical_heads,
        model_name=MODEL_NAME, k_trials=C.K_TRIALS, seed=C.MULTIHEAD_FAULT_SEED,
        num_layers=model.num_layers, progress_every=1000)
    _save(mh_df, "random_multihead.csv")
    grid = multihead.degradation_grid(mh_df, DEG_COL)
    _save(grid.reset_index(), "degradation_grid.csv")
    plots.degradation_grid_plot(
        grid, "%s - mean %s" % (MODEL_NAME, DEG_LABEL),
        FIGURES / "degradation_grid.png", value_label=DEG_LABEL)

    # Fault severity vs degradation: plain rank correlations
    from scipy.stats import kendalltau, pearsonr, spearmanr
    corr_rows = []
    for sev in ("N_fail", "L_fault", "N_crit_fail", "W_fail", "W_fail_mean"):
        x, y = mh_df[sev].to_numpy(float), mh_df[DEG_COL].to_numpy(float)
        corr_rows.append({"model": MODEL_NAME, "severity": sev, "degradation": DEG_COL,
                          "pearson": pearsonr(x, y).statistic,
                          "spearman": spearmanr(x, y).statistic,
                          "kendall": kendalltau(x, y).statistic})
    corr_df = pd.DataFrame(corr_rows)
    _save(corr_df, "severity_correlations.csv")
    print("    spearman: N_fail %.3f | W_fail %.3f | W_fail_mean %.3f"
          % tuple(corr_df.set_index("severity").loc[["N_fail", "W_fail", "W_fail_mean"],
                                                    "spearman"]), flush=True)

    # STEP 6 - empirical safe regions
    grids, safe_summary = saferegion.run_safe_region_analysis(
        mh_df, SAFE_COL, TAUS, n_heads, len(critical_heads),
        min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB,
        tau_labels=TAU_LABELS)
    _save(grids, "safe_region_grid.csv")
    _save(safe_summary, "safe_region_summary.csv")
    tau_df = saferegion.required_tau_table(
        mh_df, SAFE_COL, n_heads, len(critical_heads),
        min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB)
    _save(tau_df, "required_tau.csv")
    plots.required_tau_plot(tau_df, "%s - required threshold for 95%% safety" % MODEL_NAME,
                            FIGURES / "required_tau.png", ylabel="tau required (%s)" % SAFE_LABEL)
    print(safe_summary.to_string(index=False), flush=True)

    # summary.json
    summary = {
        "model_name": MODEL_NAME, "n_heads": n_heads, "n_layers": model.num_layers,
        "k_trials": int(C.K_TRIALS), "n_fault_eval_samples": int(C.N_FAULT),
            "clean_metrics": clean_eval, "single_head": model_summary,
        "critical_heads": ["L%d_H%d" % h for h in critical_heads],
        "criticality_col": CRIT_COL, "degradation_col": DEG_COL,
        "safe_degradation_col": SAFE_COL,
        "severity_correlations": corr_rows,
        "criticality_profile": sorted(sh_df["criticality"].tolist(), reverse=True),
        "safe_regions": safe_summary.to_dict(orient="records"),
    "criticality_profile": sorted(sh_df["criticality"].tolist(), reverse=True),
        "median_over_max_criticality": float(sh_df["criticality"].median() / (sh_df["criticality"].max() + 1e-12)),
        "n_heads_zero_effect": int((sh_df["criticality"] <= 1e-6).sum()),
        "worst_single_head": {
            "head": sh_df.loc[sh_df["criticality"].idxmax(), "head_name"],
            "criticality": float(sh_df["criticality"].max()),
            "degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), DEG_COL]),
            "safe_degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), SAFE_COL]),
        },
        **EXTRA_META,
    }
    (OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    return summary

print("analyse_region defined")

analyse_region defined


## Run every region

In [7]:
summaries = []
t_all = time.time()
for region in REGIONS:
    run_region(region)
    summaries.append(analyse_region(region))
    (RESULTS_DIR / "all_regions_summary.json").write_text(
        json.dumps(summaries, indent=2, default=str), encoding="utf-8")
    print("  [%d/%d done, %.1f min elapsed]" % (len(summaries), len(REGIONS),
                                                (time.time() - t_all) / 60.0), flush=True)
print("all regions finished in %.1f min" % ((time.time() - t_all) / 60.0))


PJM region PJME


  obs=145362 (2002-01-01..2018-08-03)  train=36000 val=4000 test=10000


    epoch  1  train_mse 0.088393  val_mae 0.042370  val_rmse 0.052594  (11s)


    epoch  2  train_mse 0.004204  val_mae 0.102014  val_rmse 0.104835  (11s)


    epoch  3  train_mse 0.002894  val_mae 0.020122  val_rmse 0.024515  (11s)


    epoch  4  train_mse 0.000781  val_mae 0.021284  val_rmse 0.025409  (11s)


... (514 outputs left out)

    spearman: N_fail 0.868 | W_fail 0.954 | W_fail_mean 0.538


     tau             safe_boundary  n_fail_max  n_crit_fail_max  max_p_safe  coverage_percent  smallest_required_tau  n_boundaries_evaluated    threshold_name
0.139187                       NaN         NaN              NaN    0.408397              0.00               0.479781                     180 10%_of_clean_mape
0.347968                       NaN         NaN              NaN    0.839695              0.00               0.479781                     180 25%_of_clean_mape
0.695935 N_fail<=1, N_crit_fail<=0         1.0              0.0    1.000000              2.62               0.479781                     180 50%_of_clean_mape


  [11/11 done, 221.4 min elapsed]


all regions finished in 221.4 min


## Cross-region comparison

In [8]:
rows = []
for s in summaries:
    sh = s["single_head"]
    rows.append({
        "region": s["region"],
        "clean_mae": s["clean_metrics"]["mae"],
        "clean_mape": s["clean_metrics"]["mape"],
        "most_critical_head": sh["most_critical_head"],
        "max_criticality": sh["max_criticality"],
        "median_criticality": sh["median_criticality"],
        "median_over_max": s["median_over_max_criticality"],
        "mean_redundancy": sh["mean_redundancy"],
        "fragile_layer_by_mean": sh["most_fragile_layer_by_mean"],
        "fragile_layer_by_max": sh["most_fragile_layer_by_max"],
        "critical_heads": ";".join(s["critical_heads"]),
        "critical_layers": ";".join(sorted({h.split("_")[0] for h in s["critical_heads"]})),
        "worst_head_delta_mape": s["worst_single_head"]["safe_degradation"],
    })
cross = pd.DataFrame(rows)
cross.to_csv(RESULTS_DIR / "cross_region_comparison.csv", index=False)
print("wrote cross_region_comparison.csv")
cross

wrote cross_region_comparison.csv


,region,clean_mae,clean_mape,most_critical_head,max_criticality,median_criticality,median_over_max,mean_redundancy,fragile_layer_by_mean,fragile_layer_by_max,critical_heads,critical_layers,worst_head_delta_mape
0,PJME,0.006990,1.051026,L0_H1,0.026234,0.001997,0.076137,0.849809,0,0,L0_H1;L0_H4;L0_H3;L2_H2,L0;L2,3.926492
1,COMED,0.008109,1.171138,L0_H3,0.021340,0.002012,0.094305,0.835934,0,0,L0_H3;L3_H1;L1_H2;L1_H0,L0;L1;L3,2.632967
2,DAYTON,0.008882,1.215294,L3_H1,0.037069,0.002395,0.064601,0.888886,3,3,L3_H1;L0_H1;L1_H2;L2_H2,L0;L1;L2;L3,5.321041
3,DOM,0.008558,1.618221,L1_H5,0.016837,0.002028,0.120423,0.793743,0,1,L1_H5;L0_H3;L1_H1;L0_H4,L0;L1,3.091130
4,DUQ,0.011302,1.367323,L1_H2,0.029051,0.003866,0.133091,0.817751,1,1,L1_H2;L3_H1;L1_H5;L1_H0,L1;L3,3.484279
5,PJMW,0.007562,1.259298,L0_H1,0.025814,0.001398,0.054163,0.861400,0,0,L0_H1;L0_H3;L1_H5;L2_H2,L0;L1;L2,4.428680
6,FE,0.008790,1.578966,L1_H5,0.031290,0.002054,0.065655,0.884669,1,1,L1_H5;L0_H3;L0_H4;L0_H5,L0;L1,5.637160
7,NI,0.009359,1.331861,L1_H5,0.042380,0.003325,0.078448,0.879193,1,1,L1_H5;L1_H0;L1_H2;L0_H3,L0;L1,6.056039
8,DEOK,0.010381,1.545140,L1_H5,0.028725,0.003549,0.123556,0.839858,1,1,L1_H5;L3_H1;L0_H4;L0_H3,L0;L1;L3,4.293506
9,EKPC,0.010904,2.255996,L0_H1,0.019226,0.001203,0.062561,0.845422,0,0,L0_H1;L0_H2;L0_H3;L1_H5,L0;L1,3.898679


### Is the critical-head location stable across regions?

In [9]:
# Is the location of the critical heads stable across regions?
print("most critical head, by region:")
print(cross[["region", "most_critical_head", "max_criticality"]].to_string(index=False))
print()
print("layer of the strongest single head:")
print(cross["fragile_layer_by_max"].value_counts().sort_index().to_string())
print()
print("layers containing critical heads (top 10%):")
print(cross["critical_layers"].value_counts().to_string())

most critical head, by region:
  region most_critical_head  max_criticality
    PJME              L0_H1         0.026234
   COMED              L0_H3         0.021340
  DAYTON              L3_H1         0.037069
     DOM              L1_H5         0.016837
     DUQ              L1_H2         0.029051
    PJMW              L0_H1         0.025814
      FE              L1_H5         0.031290
      NI              L1_H5         0.042380
    DEOK              L1_H5         0.028725
    EKPC              L0_H1         0.019226
PJM_Load              L3_H1         0.039939

layer of the strongest single head:
fragile_layer_by_max
0    4
1    5
3    2

layers containing critical heads (top 10%):
critical_layers
L0;L1          4
L0;L1;L3       2
L0;L2          1
L0;L1;L2;L3    1
L1;L3          1
L0;L1;L2       1
L1;L2;L3       1
